# Encoder vs Vicon IK — knee angle & causal angular velocity

Compare **motor encoder** (stand-in for knee angle) against **Vicon IK** on GPIO-synced telemetry time.

- **Sync**: same falling-edge GPIO alignment as the batch GT notebook (`t_aligned = t_npz + offset_s`)
- **Encoder corrections**: same per-trial xcorr lag / offset as `compare_processed_knee_exo_id.ipynb`
- **Angle**: interpolate Vicon IK and encoder onto `t_aligned`, report RMSE / R² (Vicon = reference)
- **Velocity**: causal backward `dθ/dt` on each synced angle series (real-time safe)
- **Default**: AB01 Jinwoo RA & RD (`0.8 m/s`); subject is swappable below


In [9]:
import io
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
PROCESSED_ROOT = Path('/media/metamobility3/Samsung_T52/Results/processed')
TELEMETRY_ROOT = PROJECT_ROOT

EXO_KIND = 'knee-exo'
JOINT_COL = 'knee_angle_r'
MOCAP_FS_HZ = 1000.0
TELEMETRY_PATTERN = '*_knee_*_exo_on.npz'
DEFAULT_SPEED = '0p8mps'
DEFAULT_TASKS = ('RA', 'RD')

SUBJECT_TOKEN_TO_DIR = {
    'ab01_jinwoo': 'AB01_Jinwoo', 'ab02_oscar': 'AB02_Oscar', 'ab03_ilseung': 'AB03_Ilseung',
    'ab04_changseob': 'AB04_Changseob', 'ab05_maria': 'AB05_Maria', 'ab06_jimin': 'AB06_Jimin',
    'ab07_amy': 'AB07_Amy', 'ab08_seokhyun': 'AB08_Seokhyun',
}

TRIM_START_SEC = 10.0
TRIM_END_SEC = 10.0

# Swappable subject (also editable via dropdown in the explorer cell).
SUBJECT_TOKEN = 'ab08_seokhyun'

print(f'Processed root: {PROCESSED_ROOT}')


# Per-trial encoder corrections (same as compare_processed_knee_exo_id.ipynb).
ENCODER_OFFSET_DEG_BY_STEM = {
    'ab06_jimin_knee_0p8mps_ra_exo_on': 10.0,
    'ab02_oscar_knee_0p8mps_ra_exo_on': -20.0,
    'ab08_seokhyun_knee_0p8mps_ra_exo_on': -6.0,
}
ENCODER_XCORR_ALIGN_STEMS = frozenset({
    'ab01_jinwoo_knee_0p8mps_rd_exo_on',
    'ab02_oscar_knee_0p8mps_ra_exo_on',
    'ab03_ilseung_knee_0p8mps_rd_exo_on',
    'ab05_maria_knee_0p8mps_ra_exo_on',
    'ab05_maria_knee_0p8mps_rd_exo_on',
    'ab07_amy_knee_0p8mps_ra_exo_on',
    'ab07_amy_knee_0p8mps_rd_exo_on',
    'ab08_seokhyun_knee_0p8mps_ra_exo_on',
    'ab08_seokhyun_knee_0p8mps_rd_exo_on',
})
ENCODER_EXCLUDED_STEMS = {'ab02_oscar_knee_0p8mps_rd_exo_on': 'broken encoder data'}
ENCODER_XCORR_MAX_LAG_SAMPLES = 300

print(f'Default subject: {SUBJECT_TOKEN_TO_DIR[SUBJECT_TOKEN]}')


Processed root: /media/metamobility3/Samsung_T52/Results/processed
Default subject: AB08_Seokhyun


In [10]:
def rmse_r2(y_ref: np.ndarray, y_cmp: np.ndarray) -> Tuple[float, float]:
    """RMSE and R² of y_cmp vs y_ref (reference = Vicon)."""
    m = np.isfinite(y_ref) & np.isfinite(y_cmp)
    if m.sum() < 2:
        return np.nan, np.nan
    ref = y_ref[m]
    cmp = y_cmp[m]
    err = cmp - ref
    rmse = float(np.sqrt(np.mean(err ** 2)))
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((ref - np.mean(ref)) ** 2))
    return rmse, float(1.0 - ss_res / (ss_tot + 1e-12))


def read_sto(path: Path):
    with open(path) as f:
        lines = f.readlines()
    end_idx = next(i for i, l in enumerate(lines) if l.strip().lower() == 'endheader')
    cols = lines[end_idx + 1].strip().split()
    data = np.loadtxt(io.StringIO(''.join(lines[end_idx + 2:])))
    if data.ndim == 1:
        data = data.reshape(1, -1)
    return cols, data


def infer_fs_hz(time_s, default_fs=100.0):
    if time_s is None or len(time_s) < 3:
        return float(default_fs)
    dt = np.diff(np.asarray(time_s, dtype=np.float64))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    return float(1.0 / np.median(dt)) if dt.size else float(default_fs)


def _subject_token(stem: str) -> str:
    return '_'.join(stem.lower().split('_')[:2])


def subject_dir_from_stem(stem: str) -> Path:
    token = _subject_token(stem)
    p = PROCESSED_ROOT / SUBJECT_TOKEN_TO_DIR[token]
    if not p.is_dir():
        raise FileNotFoundError(p)
    return p


def trial_cond_speed(stem: str) -> Tuple[str, str]:
    parts = stem.lower().split('_')
    return parts[4].upper(), parts[3]


def trial_stem(subject_token: str, task: str, speed: str = DEFAULT_SPEED) -> str:
    return f'{subject_token}_knee_{speed}_{task.lower()}_exo_on'


def normalize_gpio(gpio: np.ndarray) -> np.ndarray:
    g = np.asarray(gpio, dtype=np.float64)
    if g.max() > 1.5:
        g = g / np.nanmax(g)
    return g


def first_falling_edge(signal: np.ndarray, threshold: float = 0.5) -> Optional[int]:
    s = normalize_gpio(signal)
    for i in range(1, len(s)):
        if s[i - 1] >= threshold and s[i] < threshold:
            return i
    return None


def extract_gpio(npz) -> Tuple[np.ndarray, str]:
    for key in ('gpio', 'GPIO', 'jet'):
        if key in npz.files:
            return np.asarray(npz[key], dtype=np.float64), key
    raise KeyError(f'No GPIO key; available: {sorted(npz.files)}')


def parse_mocap_csv(path: Path, fs: float = MOCAP_FS_HZ):
    df = pd.read_csv(path, skiprows=[0, 1, 2, 4], header=0, low_memory=False, on_bad_lines='skip')
    df = df[pd.to_numeric(df['Frame'], errors='coerce').notna()].copy()
    df['jet'] = pd.to_numeric(df['jet'], errors='coerce')
    df = df.dropna(subset=['jet']).reset_index(drop=True)
    t_mocap = np.arange(len(df), dtype=np.float64) / float(fs)
    return t_mocap, df['jet'].to_numpy(dtype=np.float64)


def gpio_offset_s(t_exo, g_exo, t_mocap, g_mocap):
    idx_exo = first_falling_edge(g_exo)
    idx_mocap = first_falling_edge(normalize_gpio(g_mocap))
    if idx_exo is None or idx_mocap is None:
        return None, idx_exo, idx_mocap
    return float(t_mocap[idx_mocap] - t_exo[idx_exo]), idx_exo, idx_mocap


def resolve_trial_paths(trial_stem: str) -> Dict[str, Path]:
    cond, speed = trial_cond_speed(trial_stem)
    subj_dir = subject_dir_from_stem(trial_stem)
    return {
        'npz': TELEMETRY_ROOT / f'{trial_stem}.npz',
        'mocap': subj_dir / EXO_KIND / 'mocap' / f'{cond}_{speed}.csv',
        'ik': subj_dir / EXO_KIND / 'ik' / f'{cond}_{speed}_ik.mot',
        'cond': cond,
        'speed': speed,
        'subject_dir': subj_dir,
    }


def _fill_nan_1d(x: np.ndarray) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.all() or not finite.any():
        return arr
    arr[~finite] = np.interp(np.flatnonzero(~finite), np.flatnonzero(finite), arr[finite])
    return arr


def sync_to_wave_t(
    t_src: np.ndarray,
    y_src: np.ndarray,
    wave: Dict,
    *,
    t_src_on_npz_clock: bool = False,
) -> np.ndarray:
    t_aligned = np.asarray(wave['t'], dtype=np.float64)
    y_src = np.asarray(y_src, dtype=np.float64)
    t_src = np.asarray(t_src, dtype=np.float64)
    if t_src_on_npz_clock:
        n = int(min(len(t_aligned), len(t_src), len(y_src)))
        t_aligned = t_aligned[:n]
        t_src = t_src[:n]
        y_src = y_src[:n]
        t_ref = t_src + float(wave['offset_s'])
    else:
        t_ref = t_src
    y_sync = np.interp(t_aligned, t_ref, y_src, left=np.nan, right=np.nan)
    return _fill_nan_1d(y_sync)


def load_gpio_sync_wave(trial_stem: str) -> Dict:
    paths = resolve_trial_paths(trial_stem)
    for key in ('npz', 'mocap', 'ik'):
        if not paths[key].exists():
            raise FileNotFoundError(paths[key])

    d = np.load(str(paths['npz']), allow_pickle=True)
    gpio, gpio_key = extract_gpio(d)
    t_raw = np.asarray(d['time'], dtype=np.float64) if 'time' in d.files else np.arange(len(gpio), dtype=np.float64)
    n = min(len(t_raw), len(gpio))
    t_raw, gpio = t_raw[:n], gpio[:n]

    t_mocap, gpio_mocap = parse_mocap_csv(paths['mocap'])
    offset_s, idx_exo, idx_mocap = gpio_offset_s(t_raw, gpio, t_mocap, gpio_mocap)
    if offset_s is None:
        raise ValueError(f'No GPIO falling edge for {trial_stem}')

    t_aligned = t_raw + float(offset_s)
    fs_hz = infer_fs_hz(t_raw)
    cond, speed = trial_cond_speed(trial_stem)
    return {
        'trial': trial_stem,
        'task': cond,
        'speed': speed,
        'subject': SUBJECT_TOKEN_TO_DIR[_subject_token(trial_stem)],
        't': t_aligned,
        'offset_s': float(offset_s),
        'fs_hz': fs_hz,
        'gpio_key': gpio_key,
        'paths': paths,
    }


def load_vicon_ik_angle(stem: str) -> Tuple[np.ndarray, np.ndarray]:
    ik_path = resolve_trial_paths(stem)['ik']
    cols, data = read_sto(ik_path)
    t_mocap = data[:, cols.index('time')].astype(np.float64)
    knee_rad = np.deg2rad(data[:, cols.index(JOINT_COL)].astype(np.float64))
    return t_mocap, knee_rad




def _shift_encoder_samples(x: np.ndarray, lag_samples: int) -> np.ndarray:
  arr = np.asarray(x, dtype=np.float64)
  out = np.full_like(arr, np.nan)
  lag = int(lag_samples)
  if lag > 0:
    if lag < len(arr):
      out[:-lag] = arr[lag:]
  elif lag < 0:
    lag = -lag
    if lag < len(arr):
      out[lag:] = arr[:-lag]
  else:
    out = arr.copy()
  return _fill_nan_1d(out)


def _encoder_vicon_best_lag_samples(enc_rad, vicon_rad, *, max_lag=ENCODER_XCORR_MAX_LAG_SAMPLES, mask=None):
  best_lag, best_score = 0, -np.inf
  enc_rad = np.asarray(enc_rad, dtype=np.float64)
  vicon_rad = np.asarray(vicon_rad, dtype=np.float64)
  for lag in range(-int(max_lag), int(max_lag) + 1):
    shifted = _shift_encoder_samples(enc_rad, lag)
    s = shifted[mask] if mask is not None else shifted
    v = vicon_rad[mask] if mask is not None else vicon_rad
    mm = np.isfinite(s) & np.isfinite(v)
    if mm.sum() < 100:
      continue
    score = float(np.corrcoef(s[mm], v[mm])[0, 1])
    if score > best_score:
      best_score, best_lag = score, lag
  return int(best_lag)


def apply_encoder_trial_corrections(stem, enc_sync_rad, wave, vicon_sync_rad=None):
  enc = np.asarray(enc_sync_rad, dtype=np.float64).copy()
  meta = {
    'encoder_offset_deg': 0.0,
    'encoder_xcorr_lag_samples': 0,
    'encoder_correction_note': '',
    'encoder_replay_excluded': stem in ENCODER_EXCLUDED_STEMS,
  }
  if meta['encoder_replay_excluded']:
    meta['encoder_correction_note'] = f"excluded: {ENCODER_EXCLUDED_STEMS[stem]}"
    return enc, meta
  trim_m = analysis_trim_mask(wave['t'][: len(enc)])
  if stem in ENCODER_XCORR_ALIGN_STEMS:
    if vicon_sync_rad is None:
      raise ValueError(f'{stem} requires vicon_sync_rad for xcorr encoder alignment')
    lag = _encoder_vicon_best_lag_samples(enc, vicon_sync_rad, mask=trim_m)
    enc = _shift_encoder_samples(enc, lag)
    meta['encoder_xcorr_lag_samples'] = lag
    fs_hz = float(wave.get('fs_hz', infer_fs_hz(wave['t'])))
    meta['encoder_correction_note'] = f'xcorr lag {lag:+d} samples ({lag / fs_hz * 1000.0:+.0f} ms)'
  offset_deg = float(ENCODER_OFFSET_DEG_BY_STEM.get(stem, 0.0))
  if offset_deg != 0.0:
    enc = enc + np.deg2rad(offset_deg)
    meta['encoder_offset_deg'] = offset_deg
    off_note = f'{offset_deg:+.0f} deg offset'
    meta['encoder_correction_note'] = (
      f"{meta['encoder_correction_note']}; {off_note}".strip('; ')
      if meta['encoder_correction_note'] else off_note
    )
  return enc, meta


def load_encoder_angle(stem: str, wave: Dict) -> Tuple[np.ndarray, str]:
    d = np.load(str(resolve_trial_paths(stem)['npz']), allow_pickle=True)
    if 'time' not in d.files:
        raise KeyError(f"No 'time' in NPZ for {stem}")
    t_npz = np.asarray(d['time'], dtype=np.float64)
    for key in ('model_in_knee_angle_raw', JOINT_COL):
        if key in d.files:
            enc_raw = np.asarray(d[key], dtype=np.float64)
            enc_sync = sync_to_wave_t(t_npz, enc_raw, wave, t_src_on_npz_clock=True)
            return enc_sync, key
    raise KeyError(f'No encoder angle in {stem}.npz')


def causal_backward_derivative(x: np.ndarray, fs_hz: float) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64)
    dt = 1.0 / float(fs_hz)
    vel = np.zeros_like(arr)
    if len(arr) > 1:
        vel[1:] = (arr[1:] - arr[:-1]) / dt
    return vel


def analysis_trim_mask(t: np.ndarray, trim_start_s=TRIM_START_SEC, trim_end_s=TRIM_END_SEC) -> np.ndarray:
    t_rel = np.asarray(t, dtype=np.float64) - np.nanmin(t)
    t_end = float(np.nanmax(t_rel))
    return (t_rel >= float(trim_start_s)) & (t_rel <= t_end - float(trim_end_s))


def build_angle_compare(stem: str) -> Dict:
    wave = load_gpio_sync_wave(stem)
    t_mocap, vicon_rad = load_vicon_ik_angle(stem)
    vicon_rad = sync_to_wave_t(t_mocap, vicon_rad, wave, t_src_on_npz_clock=False)
    enc_rad, enc_key = load_encoder_angle(stem, wave)
    enc_rad, enc_meta = apply_encoder_trial_corrections(stem, enc_rad, wave, vicon_rad)

    n = int(min(len(wave['t']), len(vicon_rad), len(enc_rad)))
    t = np.asarray(wave['t'][:n], dtype=np.float64)
    vicon_rad = vicon_rad[:n]
    enc_rad = enc_rad[:n]
    fs_hz = float(wave['fs_hz'])

    vicon_vel = causal_backward_derivative(vicon_rad, fs_hz)
    enc_vel = causal_backward_derivative(enc_rad, fs_hz)

    m = analysis_trim_mask(t)
    ang_rmse_deg, ang_r2 = rmse_r2(np.rad2deg(vicon_rad[m]), np.rad2deg(enc_rad[m]))
    vel_rmse_dps, vel_r2 = rmse_r2(np.rad2deg(vicon_vel[m]), np.rad2deg(enc_vel[m]))

    return {
        **wave,
        't': t,
        'vicon_rad': vicon_rad,
        'enc_rad': enc_rad,
        'vicon_vel_rad_s': vicon_vel,
        'enc_vel_rad_s': enc_vel,
        'encoder_key': enc_key,
        'trim_mask': m,
        'ang_rmse_deg': ang_rmse_deg,
        'ang_r2': ang_r2,
        'vel_rmse_deg_s': vel_rmse_dps,
        'vel_r2': vel_r2,
    }


print('Helpers ready.')


Helpers ready.


In [11]:
def load_subject_trials(
    subject_token: str = SUBJECT_TOKEN,
    tasks: Tuple[str, ...] = DEFAULT_TASKS,
    speed: str = DEFAULT_SPEED,
) -> Dict[str, Dict]:
    out: Dict[str, Dict] = {}
    errors: List[Tuple[str, str]] = []
    for task in tasks:
        stem = trial_stem(subject_token, task, speed)
        try:
            out[stem] = build_angle_compare(stem)
        except Exception as exc:
            errors.append((stem, str(exc)))
    print(f"Loaded {len(out)} / {len(tasks)} trials for {SUBJECT_TOKEN_TO_DIR[subject_token]}")
    if errors:
        print('Skipped:')
        for stem, msg in errors:
            print(f'  {stem}: {msg}')
    return out


def metrics_table(trial_data: Dict[str, Dict]) -> pd.DataFrame:
    rows = []
    for stem, d in sorted(trial_data.items()):
        rows.append({
            'trial': stem,
            'subject': d['subject'],
            'task': d['task'],
            'offset_s': d['offset_s'],
            'encoder_key': d['encoder_key'],
            'ang_rmse_deg': d['ang_rmse_deg'],
            'ang_r2': d['ang_r2'],
            'vel_rmse_deg_s': d['vel_rmse_deg_s'],
            'vel_r2': d['vel_r2'],
            'n_trim': int(d['trim_mask'].sum()),
        })
    return pd.DataFrame(rows)


TRIAL_DATA = load_subject_trials(SUBJECT_TOKEN)
metrics_df = metrics_table(TRIAL_DATA)
display(metrics_df)

Loaded 2 / 2 trials for AB08_Seokhyun


,trial,subject,task,offset_s,encoder_key,ang_rmse_deg,ang_r2,vel_rmse_deg_s,vel_r2,n_trim
0,ab08_seokhyun_knee_0p8mps_ra_exo_on,AB08_Seokhyun,RA,-0.235074,model_in_knee_angle_raw,15.380894,0.198916,26.239226,0.946522,5998
1,ab08_seokhyun_knee_0p8mps_rd_exo_on,AB08_Seokhyun,RD,-0.238052,model_in_knee_angle_raw,4.419265,0.949682,33.991870,0.934753,5999


In [ ]:
def plot_angle_compare(d: Dict, ax_overlay=None, ax_resid=None):
    t = d['t'] - np.nanmin(d['t'])
    m = d['trim_mask']
    vicon_deg = np.rad2deg(d['vicon_rad'])
    enc_deg = np.rad2deg(d['enc_rad'])

    if ax_overlay is None or ax_resid is None:
        fig, (ax_overlay, ax_resid) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
        own_fig = True
    else:
        own_fig = False

    ax_overlay.plot(t, vicon_deg, color='#1565C0', lw=1.2, label='Vicon IK')
    ax_overlay.plot(t, enc_deg, color='#E65100', lw=1.0, alpha=0.9, label='Encoder')
    ax_overlay.set_ylabel('Knee angle (deg)')
    ax_overlay.legend(loc='upper right')
    ax_overlay.grid(True, alpha=0.25)
    ax_overlay.set_title(
        f"{d['subject']} {d['task']} | angle RMSE={d['ang_rmse_deg']:.2f}° R²={d['ang_r2']:.3f} | "
        f"offset={d['offset_s']:+.3f}s"
    )

    resid = enc_deg - vicon_deg
    ax_resid.plot(t[m], resid[m], color='#6A1B9A', lw=1.0)
    ax_resid.axhline(0.0, color='k', lw=0.8, alpha=0.5)
    ax_resid.set_ylabel('Encoder − Vicon (deg)')
    ax_resid.set_xlabel('Time (s)')
    ax_resid.grid(True, alpha=0.25)

    if own_fig:
        fig.tight_layout()
        plt.show()


def plot_velocity_compare(d: Dict, ax_overlay=None, ax_resid=None):
    t = d['t'] - np.nanmin(d['t'])
    m = d['trim_mask']
    vicon_dps = np.rad2deg(d['vicon_vel_rad_s'])
    enc_dps = np.rad2deg(d['enc_vel_rad_s'])

    if ax_overlay is None or ax_resid is None:
        fig, (ax_overlay, ax_resid) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
        own_fig = True
    else:
        own_fig = False

    ax_overlay.plot(t, vicon_dps, color='#1565C0', lw=1.2, label='Vicon IK dθ/dt')
    ax_overlay.plot(t, enc_dps, color='#E65100', lw=1.0, alpha=0.9, label='Encoder dθ/dt')
    ax_overlay.set_ylabel('Angular vel (deg/s)')
    ax_overlay.legend(loc='upper right')
    ax_overlay.grid(True, alpha=0.25)
    ax_overlay.set_title(
        f"{d['subject']} {d['task']} | vel RMSE={d['vel_rmse_deg_s']:.2f}°/s R²={d['vel_r2']:.3f} | "
        f"causal backward derivative"
    )

    resid = enc_dps - vicon_dps
    ax_resid.plot(t[m], resid[m], color='#6A1B9A', lw=1.0)
    ax_resid.axhline(0.0, color='k', lw=0.8, alpha=0.5)
    ax_resid.set_ylabel('Encoder − Vicon (deg/s)')
    ax_resid.set_xlabel('Time (s)')
    ax_resid.grid(True, alpha=0.25)

    if own_fig:
        fig.tight_layout()
        plt.show()


def plot_trial_panel(d: Dict):
    fig, axs = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
    plot_angle_compare(d, ax_overlay=axs[0], ax_resid=axs[1])
    plot_velocity_compare(d, ax_overlay=axs[2], ax_resid=axs[3])
    axs[3].set_xlabel('Time (s)')
    fig.suptitle(f"{d['subject']} {d['task']} — encoder vs Vicon IK (GPIO-synced)", y=1.01)
    fig.tight_layout()
    plt.show()


subject_dd = widgets.Dropdown(
    options=[(v, k) for k, v in SUBJECT_TOKEN_TO_DIR.items()],
    value=SUBJECT_TOKEN,
    description='Subject:',
)
trial_dd = widgets.Dropdown(description='Trial:')
reload_btn = widgets.Button(description='Reload', button_style='info')
out = widgets.Output()


def _trial_options(data: Dict[str, Dict]):
    return [(f"{d['task']} ({stem.split('_')[3]})", stem) for stem, d in sorted(data.items())]


def _refresh_trial_dropdown(data: Dict[str, Dict]):
    opts = _trial_options(data)
    trial_dd.options = opts
    if opts:
        trial_dd.value = opts[0][1]


def _draw(_=None):
    with out:
        out.clear_output(wait=True)
        if trial_dd.value not in TRIAL_DATA:
            print('No trial selected.')
            return
        plot_trial_panel(TRIAL_DATA[trial_dd.value])


def _reload(_=None):
    global TRIAL_DATA, metrics_df
    with out:
        out.clear_output(wait=True)
        TRIAL_DATA = load_subject_trials(subject_dd.value)
        metrics_df = metrics_table(TRIAL_DATA)
        display(metrics_df)
        _refresh_trial_dropdown(TRIAL_DATA)
        if TRIAL_DATA:
            plot_trial_panel(TRIAL_DATA[trial_dd.value])


reload_btn.on_click(_reload)
trial_dd.observe(lambda ch: _draw(), names='value')

_refresh_trial_dropdown(TRIAL_DATA)
display(widgets.HBox([subject_dd, trial_dd, reload_btn]), out)
if TRIAL_DATA:
    _draw()

Output()